# SQD Solver with IBM Quantum Backend

This notebook demonstrates how to use the SQD (Sample-based Quantum Diagonalization) solver with IBM Quantum hardware on example of N2/6-31G system. This simulation is more expensive than other tests and the execution time will be longer.

## Overview

The SQD algorithm combines:
1. **Quantum Sampling**: Build and execute LUCJ ansatz on quantum hardware
2. **Classical Post-Processing**: Diagonalize subspaces using SBD solver

## Prerequisites

Before running this notebook:
1. Create `qpu_config.yaml` from the template
2. Add your IBM Quantum credentials
3. Ensure SBD solver is installed (for post-processing)

## Setup

In [3]:
import numpy as np
from pyscf import gto, scf, mcscf, ao2mo

# Import quantum fragment methods
from quantum_fragment_methods.qpu import IBMQuantumBackend
from quantum_fragment_methods.application.solvers.quantum_zoo.sqd import SQDSolver
from quantum_fragment_methods.config import load_qpu_config

## 1. Define Molecular System

We'll use a simple N2 molecule as an example.

In [4]:
# Define N2 molecule
mol = gto.Mole()

mol.atom = '''
N 0.0 0.0 0.0
N 1.0 0.0 0.0
'''
mol.basis = '6-31G'

mol.build()

print(f"Number of orbitals: {mol.nao}")
print(f"Number of electrons: {mol.nelectron}")

Number of orbitals: 18
Number of electrons: 14


## 2. Run Mean-Field Calculation

Perform Hartree-Fock to get initial guess.

In [5]:
# Hartree-Fock calculation
mf = scf.RHF(mol)
mf.kernel()

print(f"HF energy: {mf.e_tot:.8f}")

converged SCF energy = -108.835236570774
HF energy: -108.83523657
HF energy: -108.83523657


## 3. Get Hamiltonian Integrals

In [6]:
# Define active space. Here we are freezing the 1s orbitals of both nitrogen atoms to make test more cost-efficient.
# Note the frozen core function is only a demonstration for unfragmented SQD simulations.
# In EWF-SQD simulations we are treating all of the orbitals and electrons within the system. 
# Selection of specific subset of orbitals is possible, but it follows different syntax as described in Vayesta package documentation.
n_frozen = 2
active_space = range(n_frozen, mol.nao_nr())

# Extract Hamiltonian components in MO basis
norb = len(active_space)
nelec = int(sum(mf.mo_occ[active_space]))
num_elec_a = (nelec + mol.spin) // 2
num_elec_b = (nelec - mol.spin) // 2
cas = mcscf.CASCI(mf, norb, (num_elec_a, num_elec_b))

# The `solve_from_integrals()` method in `sqd.py` (used by the EWF workflow) already expects MO-basis integrals.
mo = cas.sort_mo(active_space, base=0)
h1e, nuc_energy = cas.get_h1cas(mo)
h2e = ao2mo.restore(1, cas.get_h2cas(mo), norb)

# Nuclear repulsion energy needs to be added to SQD results at the end of calculations.
# This change is specific to SQD solver in EWF and deviates from standard Qiskit Addon SQD.

print(f"Number of orbitals: {norb}")
print(f"Total number of electrons: {nelec}")
print(f"Number of alpha electrons: {num_elec_a}")
print(f"Number of beta electrons: {num_elec_b}")
print(f"One-body tensor shape: {h1e.shape}")
print(f"Two-body tensor shape: {h2e.shape}")
print(f"Nuclear repulsion energy: {nuc_energy:.8f}")

Number of orbitals: 16
Total number of electrons: 10
Number of alpha electrons: 5
Number of beta electrons: 5
One-body tensor shape: (16, 16)
Two-body tensor shape: (16, 16, 16, 16)
Nuclear repulsion energy: -76.23110254


## 4. Load Configuration

Load QPU and SQD configuration from `config.yaml`.

The config file contains:
- **qpu**: IBM Quantum backend settings and credentials
- **sqd**: SQD solver parameters (LUCJ, transpilation, algorithm settings, SBD)
- **ext_sqd**: Extended SQD configuration

In [5]:
import yaml
from pathlib import Path

# Load configuration from config.yaml in the same directory
# For container workflows, this would be mounted at /workspace/config.yaml
# Create local copy of config_N2_demo.yaml to add you IBM Quantum credentials
config_path: Path = Path('/workspace/quantum-fragment-methods/examples/local_demo/config_local.yaml')

# Load configuration
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

# Extract configurations
qpu_config = config['qpu']
sqd_config = config['sqd']

print(f"✓ Configuration loaded from: {config_path.absolute()}")
print(f"\nQPU Settings:")
print(f"  Provider: {qpu_config['provider']}")
print(f"  Backend: {qpu_config['backend_name']}")
print(f"  Shots: {qpu_config['sampler_options']['default_shots']:,}")
print(f"\nSQD Settings:")
print(f"  Iterations: {sqd_config['iterations']}")
print(f"  Batches: {sqd_config['n_batches']}")
print(f"  Samples per batch: {sqd_config['samples_per_batch']:,}")

✓ Configuration loaded from: /workspace/quantum-fragment-methods/examples/local_demo/config_local.yaml

QPU Settings:
  Provider: ibm_cleveland
  Backend: ibm_cleveland
  Shots: 20,000

SQD Settings:
  Iterations: 5
  Batches: 1
  Samples per batch: 1,000


## 5. Initialize IBM Quantum Backend

Connect to IBM Quantum hardware using the configuration.

In [6]:
# Initialize backend (only run after adding credentials above)
backend = IBMQuantumBackend(qpu_config)
backend.initialize()
backend.get_backend()

# Display backend properties
props = backend.get_backend_properties()
print(f"Backend: {props['backend_name']}")
print(f"Number of qubits: {props['num_qubits']}")
print(f"Basis gates: {props['basis_gates']}")

qiskit_runtime_service._discover_account:WARNING:2026-05-29 22:14:29,999: Loading account with the given token. A saved account will not be used.


Backend: ibm_cleveland
Number of qubits: 156
Basis gates: ['cz', 'id', 'rz', 'sx', 'x']


## 5. Configure SQD Solver

In [7]:
# Extract SQD solver configuration from config.yaml
sqd_config = config['sqd']

print("SQD Configuration:")
print(f"  Iterations: {sqd_config['iterations']}")
print(f"  Batches: {sqd_config['n_batches']}")
print(f"  Samples per batch: {sqd_config['samples_per_batch']}")

# Create SQD solver
solver = SQDSolver(backend, config=sqd_config)
print(f"\nInitialized {solver.name} solver")

SQD Configuration:
  Iterations: 5
  Batches: 1
  Samples per batch: 1000

Initialized SQD solver


## 6. Run SQD Calculation

This will:
1. Build LUCJ circuit from CCSD amplitudes
2. Submit to IBM Quantum hardware
3. Retrieve measurement counts
4. Run SBD post-processing

In [8]:
# Run SQD solver
# Note: This will submit a job to IBM Quantum and may take time to complete
from pyscf import cc

# Compute CCSD amplitudes with frozen core to match the active space
ccsd = cc.CCSD(mf, frozen=n_frozen).run()
t1, t2 = ccsd.t1, ccsd.t2
print(f"CCSD t1 shape: {t1.shape}, t2 shape: {t2.shape}")
print(f"Expected: t2 should have dimensions based on norb={norb} active orbitals")

# Use absolute path so results are saved next to this notebook
workflow_dir = '/workspace/quantum-fragment-methods/examples/local_demo/sqd_results'

result = solver.solve(
    h1e=h1e,
    h2e=h2e,
    norb=norb,
    nelec=(num_elec_a, num_elec_b),
    t1=t1,
    t2=t2,
    mf=mf,
    workflow_path=workflow_dir,
    force_resubmit=False # Set to True to force resubmission of the job. 
    # This option must be used when starting the SQD simulation for new molecular system after performing another SQD run.
    # In case of force_resubmit=False the SQD simulation attempts to reuse previously obtained count_dict.npy in "sqd_results" directory.
    # Set force_resubmit=False if your job went through a queue and created count_dict.npy corresponding to current system.
)

print(f"\nSQD Results:")
print(f"Energy: {result.energy:.8f}")
print(f"Metadata: {result.metadata}")

# Note that for compatibility with EWF framework the nuclear energy is not included in SQD solver, 
# but instead is added at the end of SQD calculations. All energies reported in cell bellow are electronic energies only.

E(CCSD) = -109.0398256926544  E_corr = -0.2045891218799085
CCSD t1 shape: (5, 11), t2 shape: (5, 5, 11, 11)
Expected: t2 should have dimensions based on norb=16 active orbitals
CCSD t1 shape: (5, 11), t2 shape: (5, 5, 11, 11)
Expected: t2 should have dimensions based on norb=16 active orbitals
Iteration 1: Handling 1 batches
####################
Processing batch 1/1...
Iteration 1: Handling 1 batches
####################
Processing batch 1/1...
####################
Completed batches
####################
####################
Completed batches
####################
Iteration 2: Handling 1 batches
####################
Processing batch 1/1...
Iteration 2: Handling 1 batches
####################
Processing batch 1/1...
####################
Completed batches
####################
Energy change: 1.34e-01, Occupancy change: 1.21e-02
####################
Completed batches
####################
Energy change: 1.34e-01, Occupancy change: 1.21e-02
Iteration 3: Handling 1 batches
####################


## 7. CASCI Comparison

For validation, let's compare the SQD result with classical CASCI.

In [ ]:
# CASCI simulations for this system are lengthy, so we include N2/6-31G CASCI results from qiskit-addon-sqd tutorial:
# https://qiskit.github.io/qiskit-addon-sqd/tutorials/01_chemistry_hamiltonian.html
CASCI_energy = -109.04667800

print(f"\nClassical Results:")
print(f"HF energy:   {mf.e_tot:.8f}")
print(f"CASCI energy: {CASCI_energy:.8f}") # CASCI energy from qiskit-addon-sqd tutorial
print(f"\nSQD Results")
print(f"SQD energy:  {(result.energy+nuc_energy):.8f}")

# Calculate errors
sqd_error = abs((result.energy+nuc_energy) - CASCI_energy)
print(f"\nEnergy Comparison:")
print(f"SQD vs CASCI error: {sqd_error:.8f} Ha ({sqd_error * 627.5:.4f} kcal/mol)")


Classical Results:
HF energy:   -108.83523657
CASCI energy: -109.04667800

SQD Results
SQD energy:  -109.04558766

Energy Comparison:
SQD vs CASCI error: 0.00109034 Ha (0.6842 kcal/mol)


## Summary

This notebook demonstrated:
- Setting up IBM Quantum backend with configuration
- Creating and configuring SQD solver
- Building LUCJ circuits from CCSD amplitudes
- Submitting jobs to quantum hardware
- Monitoring job status
- Comparing SQD results with classical CASCI for validation